In [ ]:
import time
from pathlib import Path
import pandas as pd
import scanpy as sc
import anndata as ad
from matplotlib import pyplot as plt
import re
import sys
import session_info
import os

In [ ]:
plt.rcParams['figure.figsize'] = (3,3)
#plt.rcParams['figure.dpi'] = 500

In [ ]:
# Directories
sys.path.append(str(Path.cwd().resolve().parents[1]))
from config.paths import BASE_DIR
print('BASE_DIR:', BASE_DIR)

# input ref data
input_dir = BASE_DIR / "data" / "h5ad" / "03_scvi"

h5ad_out_dir = BASE_DIR / "data" / "h5ad" / "04_clustered"
h5ad_out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# input_adata = input_dir / "GSE254789.h5ad"
h5ad_out_1 = h5ad_out_dir / "GSE254789-scvi-leiden.h5ad"
h5ad_out_2 = h5ad_out_dir / "GSE254789-nonneurons.h5ad"

In [ ]:
adata = sc.read_h5ad(input_dir / "GSE254789-scvi.h5ad")

In [ ]:
adata.obs.predicted_doublets.value_counts()

In [ ]:
print('neighbors')
sc.pp.neighbors(adata, use_rep = 'X_scVI', random_state = 0) # Use latent representation to build neighbors graph
print('umap')
sc.tl.umap(adata, random_state = 0)
print('leiden')
sc.tl.leiden(adata, key_added='leiden', resolution=0.5)

In [ ]:
sc.pl.umap(adata, color = ['sample_id'], legend_fontsize = 10)
sc.pl.umap(adata, color = ['leiden'], legend_fontsize = 10)

# DE

In [ ]:
#sc.tl.rank_genes_groups(adata, groupby='leiden', method='wilcoxon')
#df = sc.get.rank_genes_groups_df(adata, group = None)
#df = df.copy()
#df = df[(df.pvals_adj < 0.05) & (df.logfoldchanges > .5)].copy()
#df.head(5)

#sc.pl.rank_genes_groups(adata, groupby = 'leiden', method = 'wilcoxon')

In [ ]:
sc.pl.umap(adata, color = ['Rbfox3', 'Th', 'Piezo1', 'Piezo2', 'Rgs5', 'Csf1r', 'Itgam', 'Trpv1', 'Col1a1', 'Sox10', 'S100b', 'Gfap', 'Apoe', 'Kcnj10', 'Gja1', 'Mbp', 'Cdh5', 'Pdgfra', 'Pdgfrb', 'total_counts', 'doublet_scores'], legend_fontsize = 10)

In [ ]:
sc.pl.umap(adata, color = ['leiden'], title = "Cluster", frameon = False, size = 2)

In [ ]:
# Number of cells
n_cells = adata.n_obs

# Median total counts per cell (transcript count)
median_total_counts = adata.obs['total_counts'].median()

# Median number of genes detected per cell
median_n_genes = adata.obs['n_genes_by_counts'].median()

# Print results
print(f"Number of cells: {n_cells:,}")
print(f"Median total counts per cell: {median_total_counts:.0f}")
print(f"Median number of detected genes per cell: {median_n_genes:.0f}")

# Clean messy clusters

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Cross-tabulate leiden clusters by sample_id
cluster_sample_counts = pd.crosstab(adata.obs['leiden'], adata.obs['sample_id'])

# Normalize to get proportions per cluster
cluster_sample_props = cluster_sample_counts.div(cluster_sample_counts.sum(axis=1), axis=0)

# Plot
cluster_sample_props.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='tab20')
plt.ylabel('Proportion of Cells')
plt.xlabel('Leiden Cluster')
plt.title('Sample Composition per Leiden Cluster')
plt.legend(title='Sample ID', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Compute composition of each leiden cluster across samples
composition = (
    adata.obs
    .groupby(['leiden', 'sample_id'])
    .size()
    .unstack(fill_value=0)
)

# Normalize to get proportions per cluster
composition = composition.div(composition.sum(axis=1), axis=0)

In [ ]:
# Initialize the 'artifact' column with default value
adata.obs['artifact'] = 'OK'

# Find clusters where any sample exceeds threshold
threshold = 0.5
bad_clusters = composition[composition.max(axis=1) > threshold].index.tolist()

# Label those clusters as "Imbalanced"
adata.obs.loc[adata.obs['leiden'].isin(bad_clusters), 'artifact'] = 'Imbalanced'

sc.pl.umap(adata, color = 'artifact')

In [ ]:
# keep pericytes
adata.obs.loc[adata.obs['leiden'] == '16', 'artifact'] = 'OK'

In [ ]:
sc.pl.umap(adata, color = ['artifact', 'sample_id', 'leiden'])

# Divide non-neuronal cells

In [ ]:
print(adata)
print('-' *40)
adata = adata[adata.obs['artifact'] == 'OK'].copy()
print(adata)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import zscore

# Step 1: Compute average Rbfox3 expression per cluster
expr = adata.to_df()['Rbfox3']
cluster_labels = adata.obs['leiden']
rbfox3_means = expr.groupby(cluster_labels).mean()

# Step 2: Compute z-score
rbfox3_z = pd.Series(zscore(rbfox3_means), index=rbfox3_means.index)

# Step 3: Visualize
plt.figure(figsize=(10, 4))
sns.barplot(x=rbfox3_z.index, y=rbfox3_z.values, palette="vlag")
plt.axhline(0, color='red', linestyle='--', label='Z = threshold')
plt.title("Z-scored Rbfox3 Expression per Cluster")
plt.ylabel("Z-score")
plt.xlabel("Leiden Cluster")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# Step 4: Label clusters based on Rbfox3 expression
adata.obs['class'] = 'Neuron'  # Default
low_clusters = rbfox3_z[rbfox3_z < 0].index.tolist()

# Keep clusters 18 and 7 as "Neuron"
excluded_clusters = ['18', '7', '15']
low_clusters = [cl for cl in low_clusters if cl not in excluded_clusters]
print("Low Rbfox3 clusters (excluding 18 and 7):", low_clusters)

adata.obs.loc[adata.obs['leiden'].isin(low_clusters), 'class'] = 'Non-neuron'

# Step 5: Visualize on UMAP
sc.pl.umap(adata, color='class')

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden', method='wilcoxon')
df = sc.get.rank_genes_groups_df(adata, group = None)
df = df.copy()
df = df[(df.pvals_adj < 0.05) & (df.logfoldchanges > .5)].copy()
df.head(5)

sc.pl.rank_genes_groups(adata, groupby = 'leiden', method = 'wilcoxon')

In [ ]:
sc.pl.umap(adata, color = ['leiden', 'Rbfox3', 'Snap25', 'Csf1r'], legend_loc = 'on data')

In [ ]:
bdata = adata[adata.obs['class'] == "Non-neuron"].copy()
bdata

In [ ]:
sc.pp.neighbors(bdata, use_rep = 'X_scVI', random_state = 0) # Use latent representation to build neighbors graph
sc.tl.umap(bdata, random_state = 0)

In [ ]:
sc.tl.leiden(bdata, key_added='leiden', resolution=0.5)

In [ ]:
sc.pl.umap(bdata, color = ['leiden'], legend_loc = 'on data')


In [ ]:
sc.pl.umap(bdata, color = ['sample_id'], legend_fontsize = 10)
sc.pl.umap(bdata, color = ['leiden', 'Rbfox3', 'Snap25', 'Sox10', 'Cdh5', 'Csf1r', 'Pdgfra', 'Pdgfrb', 'Mbp', 
                           'Kcnj10', 'P2ry12', 'Hexb', 'Tmem119', 'Itgax', 'H2-Ab1', 'Acta2', 'Ngfr', 'Ccl11'],
           legend_fontsize = 10)

In [ ]:
def subcluster_leiden_clusters(bdata, cluster_res_dict):
    """
    Subcluster specific Leiden clusters in `bdata` using the scVI latent space,
    allowing for different resolutions per cluster.

    Parameters:
    - bdata: AnnData object with 'X_scVI' and 'leiden' in .obs
    - cluster_res_dict: dict where keys are cluster labels (str), values are resolutions (float)
    """
    import scanpy as sc
    import pandas as pd

    # Ensure leiden is categorical
    if not pd.api.types.is_categorical_dtype(bdata.obs['leiden']):
        bdata.obs['leiden'] = bdata.obs['leiden'].astype('category')
    current_cats = bdata.obs['leiden'].cat.categories.tolist()

    for cl, res in cluster_res_dict.items():
        sub = bdata[bdata.obs['leiden'] == cl].copy()
        if sub.n_obs < 5:
            print(f"Skipping cluster {cl} (too few cells)")
            continue

        sc.pp.neighbors(sub, use_rep='X_scVI')
        sc.tl.leiden(sub, resolution=res)
        sub.obs['leiden'] = sub.obs['leiden'].astype(str)
        sub.obs['leiden'] = [f"{cl}_{x}" for x in sub.obs['leiden']]

        # Add new categories
        new_cats = [x for x in sub.obs['leiden'].unique() if x not in current_cats]
        if new_cats:
            current_cats.extend(new_cats)
            bdata.obs['leiden'] = bdata.obs['leiden'].cat.set_categories(current_cats)

        # Update main object
        bdata.obs.loc[sub.obs_names, 'leiden'] = sub.obs['leiden']

    return bdata

In [ ]:
# Use lower resolution for cluster 7 and higher for 8
res_dict = {'7': 0.05, '8': 0.05}
bdata = subcluster_leiden_clusters(bdata, res_dict)

In [ ]:
bdata.obs['leiden'] = bdata.obs['leiden'].astype('category')
bdata.obs['leiden'] = bdata.obs['leiden'].cat.remove_unused_categories()
sc.pl.umap(bdata, color='leiden')

In [ ]:
sc.tl.rank_genes_groups(bdata, groupby='leiden', method='wilcoxon')
sc.pl.rank_genes_groups(bdata, groupby = 'leiden', method = 'wilcoxon')

In [ ]:
#{str(i): "" for i in range(14)}

In [ ]:
cluster = {
    '0': 'Vascular_endothelial_1',
    '1': 'SGC',
    '2': 'SGC',
    '3': 'SGC',
    '4': 'SGC',
    '5': 'Schwann_cell',
    '6': 'Pericyte',
    '7_0': 'Macrophage_1',
    '7_1': 'Macrophage_2',
    '8_0': 'Fibroblast_1',
    '8_1': 'Fibroblast_2',
    '9': 'Vascular_endothelial_2',
    '10': 'Cycling SGC',
    '11': 'Angiogenic EC'
}

bdata.obs['cell_type'] = bdata.obs['leiden'].map(cluster).astype('category')

In [ ]:
sc.pl.umap(bdata, color = 'cell_type')

## Export

In [ ]:
adata.write_h5ad(h5ad_out_1, compression='gzip')

In [ ]:
bdata.write_h5ad(h5ad_out_2, compression='gzip')